Quick similarity comparison as a reference. The training_cells.h5ad embedding was creating a lot of trash so I needed a comparison because I could only submit once every 24 hours.

In [ ]:
import numpy as np
import pandas as pd

SUB_A = "submission_ztrl_1.csv"   # reference
SUB_B = "submission_bilinear_refit_oofalpha.csv"    # candidate to compare
ID_COL = None
EPS = 1e-12

EXCLUDE = {}
MAX_PERT_NUM = 60

In [8]:
a = pd.read_csv(SUB_A)
b = pd.read_csv(SUB_B)

if ID_COL is not None:
    ida = ID_COL
    idb = ID_COL
else:
    fallbacks = ["pert_id", "id", "ID", "sample_id", "row_id"]
    ida = next((c for c in fallbacks if c in a.columns), a.columns[0])
    idb = next((c for c in fallbacks if c in b.columns), b.columns[0])

a = a.rename(columns={ida: "_id"})
b = b.rename(columns={idb: "_id"})

# overlapping prediction columns
genes_a = [c for c in a.columns if c != "_id"]
genes_b = [c for c in b.columns if c != "_id"]
genes = sorted(list(set(genes_a) & set(genes_b)))
if len(genes) == 0:
    raise ValueError("No overlapping prediction columns between the two submissions.")

# drop dup ids
if a["_id"].duplicated().any():
    a = a.drop_duplicates("_id", keep="first")
if b["_id"].duplicated().any():
    b = b.drop_duplicates("_id", keep="first")

# align rows
m = a[["_id"] + genes].merge(b[["_id"] + genes], on="_id", how="inner", suffixes=("_a", "_b"))
if m.shape[0] == 0:
    raise ValueError("No overlapping IDs between the two submissions.")

# parse pert number from IDs like "pert_12"
pert_num = m["_id"].astype(str).str.extract(r"(\d+)$")[0].astype(int)

keep = (pert_num <= MAX_PERT_NUM) & (~m["_id"].isin(EXCLUDE))
m = m[keep].reset_index(drop=True)

Xa = m[[f"{g}_a" for g in genes]].to_numpy(np.float32)
Xb = m[[f"{g}_b" for g in genes]].to_numpy(np.float32)

num = np.sum(Xa * Xb, axis=1)
den = (np.sqrt(np.sum(Xa * Xa, axis=1)) * np.sqrt(np.sum(Xb * Xb, axis=1))) + EPS
cos_row = num / den

w = np.sqrt(np.sum(Xa * Xa, axis=1)) + EPS  # weight by reference magnitude

SimilarityScore = float(np.sum(w * cos_row) / np.sum(w))
print("SimilarityScore:", SimilarityScore)


SimilarityScore: 0.9675424098968506


In [ ]:
a['A1BG']

baseline: 0.7919893264770508
goat:     0.8149757385253906
ensemble: 0.883212149143219

Trash 1: 0.49797773361206055
Trash 2: 0.5302160978317261
Trash 3: 0.6584941744804382